[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuelcostac/generative-networks/blob/main/reduction/knowledge_distillation.ipynb)

This notebook is part of the [Generative Networks course](https://www.nahuelcosta.com/teaching/generative-networks).

# Knowledge Distillation for a Causal Language Model

In this tutorial, a 360M-parameter SmolLM2 **teacher** transfers its next-token distribution to a 135M-parameter **student**. We compare four models under the same evaluation protocol:

- the frozen teacher;
- the original pretrained student;
- a student trained only with hard next-token labels;
- a student trained with hard labels and the teacher's soft targets.

The goal is not to make the student equal to the teacher in 100 steps. It is to make the distillation mechanics, cost, and trade-offs measurable on a free Colab GPU.

## 0. Dependencies

The repository has a local directory named `transformers/`. Therefore, checking `find_spec("transformers")` is unsafe: it may find that directory even when the Hugging Face distribution is absent. This cell checks installed package metadata instead. If packages are installed or changed, restart the kernel before continuing if imports behave unexpectedly.

In [ ]:
from importlib import metadata
import subprocess
import sys

PINNED_PACKAGES = {
    "transformers": "5.14.1",
    "datasets": "5.0.0",
    "accelerate": "1.14.0",
}

to_install = []
for distribution, expected_version in PINNED_PACKAGES.items():
    try:
        installed_version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        to_install.append(f"{distribution}=={expected_version}")

if to_install:
    print("Installing the validated course environment:", ", ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *to_install])
else:
    print("Validated dependencies are already installed.")

for distribution, expected_version in PINNED_PACKAGES.items():
    actual_version = metadata.version(distribution)
    print(f"{distribution}: {actual_version}")
    assert actual_version == expected_version

## 1. Imports and Reproducibility

In [ ]:
import gc
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from torch.nn import functional as F
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DefaultDataCollator,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
training_dtype = torch.float16 if device.type == "cuda" else torch.float32
print(f"Using device: {device}; training dtype: {training_dtype}")

## 2. Configuration

The default run uses two 100-step training jobs and targets roughly 10–15 minutes on a free Colab GPU after downloads. Set `FAST_DEV_RUN=1` before starting the kernel to retain the full code path while reducing both jobs to two steps.

In [ ]:
fast_dev_run = os.environ.get("FAST_DEV_RUN", "0") == "1"

TEACHER_ID = "HuggingFaceTB/SmolLM2-360M"
STUDENT_ID = "HuggingFaceTB/SmolLM2-135M"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"

project_root = Path.cwd().parent if Path.cwd().name == "reduction" else Path.cwd()
output_dir = project_root / "reduction" / "outputs" / "knowledge_distillation"
output_dir.mkdir(parents=True, exist_ok=True)

sequence_length = 128
train_blocks = 32 if fast_dev_run else 2_048
validation_blocks = 8 if fast_dev_run else 128
per_device_batch_size = 2 if fast_dev_run else 4
gradient_accumulation_steps = 1 if fast_dev_run else 4
max_steps = 2 if fast_dev_run else 100
eval_steps = 1 if fast_dev_run else 25
learning_rate = 5e-5
temperature = 2.0
alpha = 0.5
generation_tokens = 8 if fast_dev_run else 48
evaluation_batches = 2 if fast_dev_run else 16

print("FAST_DEV_RUN:", fast_dev_run)
print("Artifacts:", output_dir)

## 3. Load and Pack WikiText-2

Both models belong to the same family and use a shared vocabulary. That compatibility is essential for logit distillation: index `i` must represent the same token for teacher and student. We concatenate documents with an end-of-sequence token and cut the stream into fixed, non-overlapping blocks.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)

assert tokenizer.get_vocab() == student_tokenizer.get_vocab(), (
    "Teacher and student vocabularies must be identical for token-level KL distillation."
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

raw_dataset = load_dataset(DATASET_ID, DATASET_CONFIG)

def pack_split(split, number_of_blocks):
    target_tokens = number_of_blocks * sequence_length
    token_ids = []
    for row in raw_dataset[split]:
        text = row["text"].strip()
        if not text:
            continue
        token_ids.extend(tokenizer(text, add_special_tokens=False)["input_ids"])
        token_ids.append(tokenizer.eos_token_id)
        if len(token_ids) >= target_tokens:
            break

    usable_tokens = min(len(token_ids), target_tokens)
    usable_tokens -= usable_tokens % sequence_length
    blocks = [
        token_ids[start:start + sequence_length]
        for start in range(0, usable_tokens, sequence_length)
    ]
    assert len(blocks) == number_of_blocks
    return Dataset.from_dict({
        "input_ids": blocks,
        "attention_mask": [[1] * sequence_length for _ in blocks],
        "labels": [block.copy() for block in blocks],
    })

train_dataset = pack_split("train", train_blocks)
validation_dataset = pack_split("validation", validation_blocks)
collator = DefaultDataCollator()

print(train_dataset, validation_dataset)
print(tokenizer.decode(train_dataset[0]["input_ids"][:80]))

## 4. Hard and Soft Targets

For causal language modeling, logits at position `t` predict the token at `t+1`. The supervised term is ordinary next-token cross-entropy. The distillation term compares the teacher and student distributions after dividing both logits by temperature `T`:

`loss = alpha * T^2 * KL(teacher || student) + (1 - alpha) * CE(labels, student)`

The KL term below is summed over the vocabulary and averaged only over valid shifted tokens. This avoids accidentally changing its scale with padding or sequence length.

In [ ]:
toy_teacher_logits = torch.tensor([[8.0, 4.0, 1.0]])
for toy_temperature in [1.0, 2.0, 5.0]:
    probabilities = F.softmax(toy_teacher_logits / toy_temperature, dim=-1)
    print(f"T={toy_temperature:.0f}: {probabilities.squeeze().tolist()}")

## 5. Evaluation and Benchmark Helpers

All variants use the same validation blocks and greedy prompt. Throughput is warmed up and CUDA-synchronized. It remains a local hardware measurement, not a universal ranking.

In [ ]:
def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())

def directory_size_mb(path):
    path = Path(path)
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file()) / 1e6

def release_model(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

@torch.no_grad()
def validation_loss(model, max_batches=evaluation_batches):
    model.eval()
    model_device = model.get_input_embeddings().weight.device
    loader = DataLoader(
        validation_dataset,
        batch_size=per_device_batch_size,
        shuffle=False,
        collate_fn=collator,
    )
    losses = []
    for batch_index, batch in enumerate(loader):
        if batch_index >= max_batches:
            break
        batch = {name: value.to(model_device) for name, value in batch.items()}
        losses.append(model(**batch).loss.float().item())
    return float(np.mean(losses))

@torch.no_grad()
def generation_profile(model, prompt="Knowledge distillation"):
    model.eval()
    model_device = model.get_input_embeddings().weight.device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)
    generation_kwargs = {
        "max_new_tokens": generation_tokens,
        "do_sample": False,
        "pad_token_id": tokenizer.eos_token_id,
    }
    _ = model.generate(**inputs, max_new_tokens=2, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    output = model.generate(**inputs, **generation_kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    new_tokens = output.size(1) - inputs["input_ids"].size(1)
    return tokenizer.decode(output[0], skip_special_tokens=True), new_tokens / elapsed

def profile_model(name, model, checkpoint_path=None):
    loss = validation_loss(model)
    sample, tokens_per_second = generation_profile(model)
    row = {
        "model": name,
        "parameters_m": parameter_count(model) / 1e6,
        "memory_footprint_mb": model.get_memory_footprint() / 1e6,
        "validation_loss": loss,
        "perplexity": math.exp(min(loss, 20)),
        "tokens_per_second": tokens_per_second,
        "checkpoint_mb": directory_size_mb(checkpoint_path) if checkpoint_path else np.nan,
        "sample": sample,
    }
    print(name, {key: value for key, value in row.items() if key != "sample"})
    print(sample)
    return row

## 6. Training Configuration

The hard-label and distilled students start from the exact same Hub checkpoint, use the same seed, and see the same number of optimization steps. This is the minimum fair comparison; only the objective changes.

In [ ]:
def make_training_arguments(run_name):
    return TrainingArguments(
        output_dir=str(output_dir / run_name),
        max_steps=max_steps,
        per_device_train_batch_size=per_device_batch_size,
        per_device_eval_batch_size=per_device_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        eval_strategy="steps",
        eval_steps=eval_steps,
        logging_steps=1 if fast_dev_run else 10,
        save_strategy="no",
        report_to="none",
        fp16=device.type == "cuda",
        use_cpu=device.type != "cuda",
        seed=SEED,
        data_seed=SEED,
        remove_unused_columns=False,
        dataloader_pin_memory=device.type == "cuda",
    )

## 7. Original Student and Hard-Label Baseline

In [ ]:
results = []

hard_student = AutoModelForCausalLM.from_pretrained(STUDENT_ID, dtype=training_dtype).to(device)
original_student_parameters = parameter_count(hard_student)
results.append(profile_model("Original student", hard_student))

hard_trainer = Trainer(
    model=hard_student,
    args=make_training_arguments("hard_label_trainer"),
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=collator,
)
hard_trainer.train()

hard_checkpoint = output_dir / "hard_label_student"
hard_student.save_pretrained(hard_checkpoint, safe_serialization=True)
tokenizer.save_pretrained(hard_checkpoint)
results.append(profile_model("Hard-label student", hard_student, hard_checkpoint))

assert parameter_count(hard_student) == original_student_parameters
del hard_trainer, hard_student
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8. Distillation Trainer

`Trainer` normally owns the student forward pass. Distillation also needs the teacher's logits, so we override `compute_loss`. The teacher stays in evaluation mode, is evaluated under `no_grad`, and has every parameter frozen.

In [ ]:
class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model, temperature, alpha, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.temperature = temperature
        self.alpha = alpha
        self.last_components = {}

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        student_outputs = model(**inputs)
        teacher_inputs = {
            key: value for key, value in inputs.items() if key != "labels"
        }
        with torch.no_grad():
            teacher_outputs = self.teacher_model(**teacher_inputs)

        student_logits = student_outputs.logits[:, :-1, :].float()
        teacher_logits = teacher_outputs.logits[:, :-1, :].float()
        valid_tokens = inputs["labels"][:, 1:] != -100

        student_log_probabilities = F.log_softmax(
            student_logits / self.temperature, dim=-1
        )
        teacher_probabilities = F.softmax(
            teacher_logits / self.temperature, dim=-1
        )
        token_kl = F.kl_div(
            student_log_probabilities, teacher_probabilities, reduction="none"
        ).sum(dim=-1)
        distillation_loss = token_kl[valid_tokens].mean() * self.temperature**2
        supervised_loss = student_outputs.loss.float()
        loss = (
            self.alpha * distillation_loss
            + (1.0 - self.alpha) * supervised_loss
        )
        self.last_components = {
            "supervised": supervised_loss.detach().item(),
            "distillation": distillation_loss.detach().item(),
            "total": loss.detach().item(),
        }
        return (loss, student_outputs) if return_outputs else loss

## 9. Train the Distilled Student

In [ ]:
teacher = AutoModelForCausalLM.from_pretrained(TEACHER_ID, dtype=training_dtype).to(device)
teacher.eval()
teacher.requires_grad_(False)
assert parameter_count(teacher) > original_student_parameters
assert not any(parameter.requires_grad for parameter in teacher.parameters())
results.append(profile_model("Teacher", teacher))

distilled_student = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, dtype=training_dtype
).to(device)
assert parameter_count(distilled_student) == original_student_parameters

distillation_trainer = DistillationTrainer(
    model=distilled_student,
    teacher_model=teacher,
    temperature=temperature,
    alpha=alpha,
    args=make_training_arguments("distillation_trainer"),
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=collator,
)
distillation_trainer.train()
print("Final loss components:", distillation_trainer.last_components)

assert all(math.isfinite(value) for value in distillation_trainer.last_components.values())
assert all(parameter.grad is None for parameter in teacher.parameters())

distilled_checkpoint = output_dir / "distilled_student"
distilled_student.save_pretrained(distilled_checkpoint, safe_serialization=True)
tokenizer.save_pretrained(distilled_checkpoint)
results.append(
    profile_model("Distilled student", distilled_student, distilled_checkpoint)
)

## 10. Compare the Trade-off

The two trained students have the same architecture and parameter count. Distillation changes the behavior learned by that architecture; it does not remove additional parameters during this training run. The teacher-versus-student comparison shows where deployment savings come from.

In [ ]:
results_table = pd.DataFrame(results).set_index("model")
display(results_table.drop(columns="sample").round(3))

assert results_table.loc["Teacher", "parameters_m"] > results_table.loc["Distilled student", "parameters_m"]
assert math.isfinite(results_table.loc["Distilled student", "validation_loss"])

quality_columns = ["validation_loss", "perplexity"]
efficiency_columns = ["parameters_m", "memory_footprint_mb", "tokens_per_second"]
axes = results_table[quality_columns + efficiency_columns].plot(
    kind="bar", subplots=True, layout=(2, 3), figsize=(15, 8), legend=False
)
for axis in np.asarray(axes).flat:
    axis.tick_params(axis="x", rotation=25)
plt.tight_layout()

## 11. Reload the Distilled Checkpoint

A deployment checkpoint must reconstruct independently of live trainer state.

In [ ]:
reloaded_student = AutoModelForCausalLM.from_pretrained(
    distilled_checkpoint, dtype=training_dtype
).to(device)
reloaded_student.eval()

check_inputs = tokenizer("Compression should preserve", return_tensors="pt").to(device)
with torch.no_grad():
    live_logits = distilled_student(**check_inputs).logits
    reloaded_logits = reloaded_student(**check_inputs).logits

assert torch.equal(live_logits, reloaded_logits)
print("Reloaded checkpoint logits match exactly.")

## 12. Other Forms of Distillation

This notebook uses **logit distillation**. Two important alternatives are:

- **Sequence distillation:** generate and filter teacher responses, then fine-tune a student on that text. It works when teacher logits are unavailable.
- **Representation distillation:** align hidden states or attention maps. It usually requires compatible internal shapes or learned projection layers.

A production decision should include the cost of teacher inference and student training, not only the student's final serving cost.

## 13. Things to Try

- Sweep temperatures `1`, `2`, and `4`; inspect both loss components.
- Sweep `alpha` from pure hard-label training to pure distillation.
- Distill only the teacher's top-k logits and measure the memory/quality trade-off.
- Generate teacher sequences and compare sequence distillation with token-level KL.
- Increase the training budget and repeat runs with multiple seeds before claiming an improvement.

## Troubleshooting

- If imports resolve to the local course directory, rerun the dependency cell and restart the kernel.
- If CUDA runs out of memory, lower `per_device_batch_size` before reducing sequence length.
- If training is slow, confirm a GPU runtime is active; `FAST_DEV_RUN=1` is only a pipeline check.
- If vocabularies differ, do not force the logits to align by truncation. Use sequence distillation or define an explicit token mapping.
- If KL becomes non-finite, keep its softmax/log-softmax computation in float32 as implemented here.

**References:** [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531), [SmolLM2](https://huggingface.co/HuggingFaceTB/SmolLM2-360M), and [WikiText](https://huggingface.co/datasets/Salesforce/wikitext).